# Thêm Thư Viện

In [1]:
import pyodbc
import pandas as pd
import numpy as np

# Tạo kết nối

In [2]:
conn_dwh_library = pyodbc.connect(
    'DRIVER={ODBC Driver 17 for SQL Server};'
    'SERVER=192.168.150.6;' # Địa chỉ IP của SQL Server
    'DATABASE=dwh_library;' # Tên cơ sở dữ liệu
    'UID=itc;'              # Tên đăng nhập
    'PWD=spkt@2024;')

## Đọc data từ SQL Server

In [3]:
query_phieumuon = """SELECT TL.ID_tai_lieu, ID_xep_gia, Ma_tai_lieu, TL.ID_mon, ID_ctdt, Ngay_giao_dich 
                        FROM DIM_Tai_lieu TL
                                JOIN DIM_Xep_gia XG ON TL.ID_tai_lieu =  XG.ID_tai_lieu
                                JOIN DIM_Mon M ON M.ID_mon = TL.ID_mon
                                JOIN DIM_Mon_CTDT M_CTDT ON M_CTDT.ID_mon = M.ID_mon"""
df_phieumuon = pd.read_sql(query_phieumuon, conn_dwh_library)
print(df_phieumuon)

C:\Users\phung\AppData\Local\Temp\ipykernel_9900\788839399.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_phieumuon = pd.read_sql(query_phieumuon, conn_dwh_library)


        ID_tai_lieu  ID_xep_gia  Ma_tai_lieu  ID_mon ID_ctdt  Ngay_giao_dich
0             11090      242935  SK040011093       0       0        20061206
1             11090      242936  SK040011093       0       0        20061206
2             11090      242937  SK040011093       0       0        20061206
3             11090      242938  SK040011093       0       0        20061206
4             11090      242939  SK040011093       0       0        20061206
...             ...         ...          ...     ...     ...             ...
592099        42806     1043284    SKV119272       0       0        20241113
592100        42020     1043285    SKV116547       0       0        20241113
592101        41955     1043317    SKV116029       0       0        20241115
592102        65719     1043329  SK240065766       0       0        20241115
592103        65970     1043356  SK240065982       0       0        20241121

[592104 rows x 6 columns]


# Xử lý code

## Lượng người dùng theo thư viện, nhóm bạn, ngày mượn

In [4]:
so_luong_ban_sach = df_phieumuon.groupby(['ID_tai_lieu', 'Ma_tai_lieu', 'ID_mon', 'ID_ctdt', 'Ngay_giao_dich'])['ID_xep_gia'].count().reset_index()
so_luong_ban_sach = so_luong_ban_sach.rename(columns={'ID_xep_gia': 'So_ban_sach'})
print(so_luong_ban_sach)

       ID_tai_lieu  Ma_tai_lieu  ID_mon ID_ctdt  Ngay_giao_dich  So_ban_sach
0                0            0       0       0               0            1
1               26    SKV000004       0       0        20070417            1
2               30  SK020000030       0       0        20070417            2
3               31  SK020000032       0       0        20060427            5
4               38  SK020000040       0       0        20060427            6
...            ...          ...     ...     ...             ...          ...
58525        66562  SK240066562       0       0        20241128            1
58526        66566  SK240066569       0       0        20241128            1
58527        66582  SK240066648       0       0        20241128            1
58528        66583  SK240066649       0       0        20241127            1
58529        66585  SK240066650       0       0        20241128            1

[58530 rows x 6 columns]


## Load data

### [Nếu cần] Clear bảng

In [5]:
cursor = conn_dwh_library.cursor()
truncate_query = "DELETE FROM FACT_Thong_ke_tai_lieu"
cursor.execute(truncate_query)
conn_dwh_library.commit()
cursor.close()

### Load data vào bảng Dim

In [6]:
cursor_dwh = conn_dwh_library.cursor()
insert_query = """
                INSERT INTO FACT_Thong_ke_tai_lieu (ID_tai_lieu,
                                                    Ma_xep_gia, 
                                                    ID_mon,  
                                                    ID_ctdt,
                                                    ID_date, 
                                                    So_ban_sach)
                VALUES (?, ?, ?, ?, ?, ?)"""
for index, row in so_luong_ban_sach.iterrows():
   # Trích xuất giá trị từ các cột
    values = (row['ID_tai_lieu'],
              row['Ma_tai_lieu'],
              row['ID_mon'],
              row['ID_ctdt'],
              row['Ngay_giao_dich'],
              row['So_ban_sach'])  # Nếu cột này có tên đúng
    cursor_dwh.execute(insert_query, values)
conn_dwh_library.commit()